In [ ]:
from d_CSL import GcStar

In [ ]:
# import sys
# sys.path.append(r"../")
# sys.path.append(r'C:\Users\sadiq\Desktop\U_Vienna\codes\icm-zebrafish')
import numpy as np
import matplotlib.pyplot as plt
# from d_CSL import *
from tqdm.notebook import tqdm
from reloading import reloading

In [ ]:
%reload_ext autoreload
%autoreload 2
from tryParallel import *

In [ ]:
gcstar = GcStar(n_perm=1000, n_pasts=5, n_lags=1, temporal=True)

In [ ]:
np.random.seed(3465)
n_neur=20
A = np.random.choice([0,0.5,0.85], p=[0.9,0.03,0.07], size=(n_neur,n_neur)) ### A is not the adjacency matrix in the typical sense
A = 0.5*(A)
A[0:10,0:10] =np.zeros((10,10))
for n, i in enumerate(A):
    A[n][n]=1
for n, i in enumerate(A):
    A[n]=A[n]/np.sum(A[n])
A.shape

In [ ]:
plt.imshow(A!=0)
# plt.grid()

In [ ]:
X = np.zeros([A.shape[0],5000]).T
X[0] = np.random.randn(A.shape[0])
for i, row in enumerate(X[:-1]):
    X[i+1] = A @ X[i] + 0.05*np.random.normal(0,0.25,A.shape[0])
X=X.T
plt.matshow(X, aspect='auto',vmin=0)

In [ ]:
plt.imshow(gcstar.shift_data(arr=X), aspect='auto', vmin=0)

In [ ]:
print(gcstar.get_number_of_neurons())

In [ ]:
%%time
corr, pVal_corr = gcstar.correlation_func() 

In [ ]:
fig,ax = plt.subplots(1,3,figsize=(6,6))
ax[0].imshow(corr)
ax[1].imshow(pVal_corr)
ax[2].imshow(np.multiply(corr,pVal_corr<=.01))

In [ ]:
%%time
inv_corr, pVal_inv_corr = gcstar.inv_correlation_func()

In [ ]:
fig,ax = plt.subplots(1,3,figsize=(6,6))
ax[0].imshow(inv_corr)
ax[1].imshow(pVal_inv_corr)
ax[2].imshow(np.multiply(inv_corr, pVal_inv_corr<=.001))

In [ ]:
gcstar.fit(X)

In [ ]:
plt.imshow(gcstar.get_connectivity_matrix(0.01, 0.001))

In [ ]:
p = np.transpose(np.where(gcstar.conn_mat != 0))
print(p[0])

In [ ]:
print(gcstar.compute_confusion_matrix(A.T))

In [ ]:
gcstar.compute_metrics()

In [ ]:
from d_CSL import Visualize_on_topography

In [ ]:
visualize = Visualize_on_topography(10,5,3)

In [ ]:
visualize.repopulate(np.)

In [ ]:
### test data

In [ ]:
new = np.arange(60).reshape(3,20)
print(new)

In [ ]:
qq = GcStar(100, 5, 2, True)
qq

In [ ]:
new_ = qq.shift_data(new)
new_

In [ ]:
qq.data

In [ ]:
qq.get_conditioning_set(1, 0)

In [ ]:
def prep_data(X, nn):
    if nn == None or nn == 0:
        X_ = X
        return X_
    else:
        X_ = X[:,(nn):]
        for i in range(nn):
            idx1, idx2 = nn-1-i,-i-1
            X_ = np.r_[X_, X[:,idx1:idx2]]
        return X_
        
def conditioning_set(X,n_past,i,j):
    n = X.shape[0]
    k, data = i//n, prep_data(X,n_past)
    z_ = np.delete(data,np.r_[np.arange(k*n),np.arange(j*n), [i]], axis=0)
    y_ = prep_data(X[j,:].reshape((1,X.shape[1])),n_past)
    z = np.r_[z_,y_[1:k]]
    return z

In [ ]:
def conditioning_set_(X,n_past,i,j):
    n = X.shape[0]
    k, i_ = i//n, i%n
    Z = np.delete(X, [i_, j], axis=0)
    if Z.shape[0] == 1:
        z = np.r_[prep_data(X[[i_,j], :], n_past)[k+1*n:], prep_data(Z.reshape((1,X.shape[1])), n_past)[:k*n - 2, :]]
    else:
        z = np.r_[prep_data(X[[i_,j], :], n_past)[k+1*n:], prep_data(Z,n_past)[:k*n - 2, :]]
    
    
    # else:
    #     k, data = i//n, prep_data(X,n_past)
    #     z_ = np.delete(data,np.r_[np.arange(k*n), np.arange(j*n), [i]], axis=0)
    #     y_ = prep_data(X[j,:].reshape((1,X.shape[1])),n_past)
    #     z = np.r_[z_,y_[1:k]]
    return z

In [ ]:
conditioning_set_(new,5,12,2)

In [ ]:
new